# Reto | Análisis del progreso mundial de vacunación

Dataset: *COVID-19 World Vaccination Progress* (Our World in Data, vía Kaggle).

## a. Extraer la información del archivo

Descarga del dataset con `kagglehub` y lectura de `country_vaccinations.csv` con pandas. Las cifras del reto corresponden al archivo entregado (86,512 filas); si Kaggle publica una versión nueva, pueden variar.

In [1]:
import glob, os
import pandas as pd
import numpy as np

try:
    import kagglehub
    path = kagglehub.dataset_download("gpreda/covid-world-vaccination-progress")
    archivo = glob.glob(os.path.join(path, "*.csv"))[0]
except Exception:
    # Alternativa: archivo local o subida manual en Colab
    if os.path.exists("country_vaccinations.csv"):
        archivo = "country_vaccinations.csv"
    else:
        from google.colab import files
        archivo = list(files.upload())[0]

df = pd.read_csv(archivo)
print(archivo)
df.head()

country_vaccinations.csv


,country,iso_code,date,total_vaccinations,people_vaccinated,people_fully_vaccinated,daily_vaccinations_raw,daily_vaccinations,total_vaccinations_per_hundred,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,daily_vaccinations_per_million,vaccines,source_name,source_website
0,Afghanistan,AFG,2021-02-22,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/
1,Afghanistan,AFG,2021-02-23,NaN,NaN,NaN,NaN,1367.0,NaN,NaN,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/
2,Afghanistan,AFG,2021-02-24,NaN,NaN,NaN,NaN,1367.0,NaN,NaN,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/
3,Afghanistan,AFG,2021-02-25,NaN,NaN,NaN,NaN,1367.0,NaN,NaN,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/
4,Afghanistan,AFG,2021-02-26,NaN,NaN,NaN,NaN,1367.0,NaN,NaN,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/


## b. Estructura y tipos de dato de cada columna

Se convierte `date` a `datetime64`.

In [2]:
df["date"] = pd.to_datetime(df["date"])
df.info()
print()
print(df.dtypes)

<class 'pandas.DataFrame'>
RangeIndex: 86512 entries, 0 to 86511
Data columns (total 15 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   country                              86512 non-null  str           
 1   iso_code                             86512 non-null  str           
 2   date                                 86512 non-null  datetime64[us]
 3   total_vaccinations                   43607 non-null  float64       
 4   people_vaccinated                    41294 non-null  float64       
 5   people_fully_vaccinated              38802 non-null  float64       
 6   daily_vaccinations_raw               35362 non-null  float64       
 7   daily_vaccinations                   86213 non-null  float64       
 8   total_vaccinations_per_hundred       43607 non-null  float64       
 9   people_vaccinated_per_hundred        41294 non-null  float64       
 10  people_fully_vaccinat

## c. Vacunas aplicadas de cada compañía

Se agrupa por `vaccines` (tal como lo reporta cada país) y se suma `daily_vaccinations`, que es la cantidad de dosis aplicadas por día.

In [3]:
por_compania = (df.groupby("vaccines")["daily_vaccinations"].sum()
                .sort_values(ascending=False))
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:,.0f}".format)
por_compania.to_frame("vacunas_aplicadas")

,vacunas_aplicadas
vaccines,
"CanSino, Sinopharm/Beijing, Sinopharm/Wuhan, Sinovac, ZF2001","3,250,342,496"
"Covaxin, Oxford/AstraZeneca, Sputnik V","1,829,085,487"
"Moderna, Oxford/AstraZeneca, Pfizer/BioNTech","620,164,884"
"Johnson&Johnson, Moderna, Pfizer/BioNTech","589,020,809"
"Johnson&Johnson, Moderna, Novavax, Oxford/AstraZeneca, Pfizer/BioNTech","505,216,912"
...,...
"Johnson&Johnson, Moderna","248,463"
"Oxford/AstraZeneca, Sinopharm/Beijing, Sputnik V","200,773"
"Oxford/AstraZeneca, Pfizer/BioNTech, Sputnik V","127,997"


## d. Vacunas aplicadas en todo el mundo

In [4]:
total_mundo = df["daily_vaccinations"].sum()
print(f"Vacunas aplicadas en el mundo: {total_mundo:,.0f}")

Vacunas aplicadas en el mundo: 11,320,239,871


## e. Promedio de vacunas aplicadas por país

Se suma por país y se promedia entre países.

In [5]:
por_pais = df.groupby("country")["daily_vaccinations"].sum()
print(f"Promedio por país: {por_pais.mean():,.0f}")

Promedio por país: 50,763,407


## f. Vacunas aplicadas el 29/01/21 en todo el mundo

In [6]:
dia = df[df["date"] == "2021-01-29"]["daily_vaccinations"].sum()
print(f"Vacunas aplicadas el 29/01/2021: {dia:,.0f}")

Vacunas aplicadas el 29/01/2021: 4,884,052


## g. DataFrame `conDiferencias`

Columna derivada `diferencias` = `daily_vaccinations` − `daily_vaccinations_raw`.

In [7]:
conDiferencias = df.copy()
conDiferencias["diferencias"] = (conDiferencias["daily_vaccinations"]
                                 - conDiferencias["daily_vaccinations_raw"])
conDiferencias[["country", "date", "daily_vaccinations_raw", "daily_vaccinations", "diferencias"]].dropna().head()

,country,date,daily_vaccinations_raw,daily_vaccinations,diferencias
94,Afghanistan,2021-05-27,"2,859","6,487","3,628"
101,Afghanistan,2021-06-03,"4,015","5,285","1,270"
339,Afghanistan,2022-01-27,"6,868","9,802","2,934"
397,Albania,2021-01-13,60,63,3
398,Albania,2021-01-14,78,66,-12


## h. Periodo entre el registro más reciente y el más antiguo

In [8]:
fecha_min, fecha_max = df["date"].min(), df["date"].max()
print(f"Más antiguo: {fecha_min.date()} | Más reciente: {fecha_max.date()}")
print(f"Periodo: {fecha_max - fecha_min}")

Más antiguo: 2020-12-02 | Más reciente: 2022-03-29
Periodo: 482 days 00:00:00


## i. DataFrame `conCantidad`

Columna derivada `canVac` = número de vacunas distintas utilizadas, separando `vaccines` por la coma.

In [9]:
conCantidad = df.copy()
conCantidad["canVac"] = conCantidad["vaccines"].str.split(",").str.len()
conCantidad[["country", "date", "vaccines", "canVac"]].head()

,country,date,vaccines,canVac
0,Afghanistan,2021-02-22,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing",4
1,Afghanistan,2021-02-23,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing",4
2,Afghanistan,2021-02-24,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing",4
3,Afghanistan,2021-02-25,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing",4
4,Afghanistan,2021-02-26,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing",4


## j. DataFrame `antes20`

Registros anteriores al 20 de diciembre de 2020.

In [10]:
antes20 = df[df["date"] < "2020-12-20"].copy()
print(len(antes20), "registros")
antes20.head()

70 registros


,country,iso_code,date,total_vaccinations,people_vaccinated,people_fully_vaccinated,daily_vaccinations_raw,daily_vaccinations,total_vaccinations_per_hundred,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,daily_vaccinations_per_million,vaccines,source_name,source_website
13403,Canada,CAN,2020-12-14,5,5,NaN,NaN,NaN,0,0,NaN,NaN,"Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech",Official data from provinces via covid19tracker.ca,https://covid19tracker.ca/vaccinationtracker.html
13404,Canada,CAN,2020-12-15,727,727,NaN,722,722,0,0,NaN,19,"Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech",Official data from provinces via covid19tracker.ca,https://covid19tracker.ca/vaccinationtracker.html
13405,Canada,CAN,2020-12-16,"3,025","3,025",NaN,"2,298","1,510",0,0,NaN,40,"Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech",Official data from provinces via covid19tracker.ca,https://covid19tracker.ca/vaccinationtracker.html
13406,Canada,CAN,2020-12-17,"7,279","7,279",NaN,"4,254","2,425",0,0,NaN,64,"Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech",Official data from provinces via covid19tracker.ca,https://covid19tracker.ca/vaccinationtracker.html
13407,Canada,CAN,2020-12-18,"11,296","11,296",NaN,"4,017","2,823",0,0,NaN,74,"Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech",Official data from provinces via covid19tracker.ca,https://covid19tracker.ca/vaccinationtracker.html


## k. DataFrame `pfizer`

Registros donde se usó la vacuna Pfizer (`Pfizer/BioNTech` aparece en `vaccines`).

In [11]:
pfizer = df[df["vaccines"].str.contains("Pfizer", case=False, na=False)].copy()
print(len(pfizer), "registros")
pfizer.head()

64193 registros


,country,iso_code,date,total_vaccinations,people_vaccinated,people_fully_vaccinated,daily_vaccinations_raw,daily_vaccinations,total_vaccinations_per_hundred,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,daily_vaccinations_per_million,vaccines,source_name,source_website
0,Afghanistan,AFG,2021-02-22,0,0,NaN,NaN,NaN,0,0,NaN,NaN,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing",World Health Organization,https://covid19.who.int/
1,Afghanistan,AFG,2021-02-23,NaN,NaN,NaN,NaN,"1,367",NaN,NaN,NaN,34,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing",World Health Organization,https://covid19.who.int/
2,Afghanistan,AFG,2021-02-24,NaN,NaN,NaN,NaN,"1,367",NaN,NaN,NaN,34,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing",World Health Organization,https://covid19.who.int/
3,Afghanistan,AFG,2021-02-25,NaN,NaN,NaN,NaN,"1,367",NaN,NaN,NaN,34,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing",World Health Organization,https://covid19.who.int/
4,Afghanistan,AFG,2021-02-26,NaN,NaN,NaN,NaN,"1,367",NaN,NaN,NaN,34,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing",World Health Organization,https://covid19.who.int/


## l. Almacenar los resultados en `resultadosReto.xlsx`

Cada DataFrame va en una hoja distinta con `pd.ExcelWriter`.

In [12]:
with pd.ExcelWriter("resultadosReto.xlsx", engine="xlsxwriter",
                    engine_kwargs={"options": {"strings_to_urls": False}}) as writer:
    conDiferencias.to_excel(writer, sheet_name="conDiferencias", index=False)
    conCantidad.to_excel(writer, sheet_name="conCantidad", index=False)
    antes20.to_excel(writer, sheet_name="antes20", index=False)
    pfizer.to_excel(writer, sheet_name="pfizer", index=False)

print(pd.ExcelFile("resultadosReto.xlsx").sheet_names)

# Solo en Colab: descargar el archivo
try:
    from google.colab import files
    files.download("resultadosReto.xlsx")
except ImportError:
    pass

['conDiferencias', 'conCantidad', 'antes20', 'pfizer']
